# 22: Sequence-to-Sequence with Attention

## The Translation Problem

Machine translation seems simple: input a sentence in one language, output it in another. But languages have different word orders, idioms, and structures.

**Sequence-to-sequence (Seq2Seq)** models handle this with:
1. An **encoder** that reads the input sentence
2. A **decoder** that generates the output sentence

### The Web Dev Analogy

Think of Seq2Seq like a human interpreter:
- **Without attention**: Listen to entire speech, then translate from memory (loses details)
- **With attention**: Refer back to notes while translating each part (preserves details)

Attention is like `Ctrl+F` - instead of remembering everything, you look up what you need!

## What You'll Learn
- [ ] Explain the encoder-decoder architecture for sequence-to-sequence tasks
- [ ] Implement cross-attention between encoder and decoder
- [ ] Understand how attention improves translation quality

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 17**: RNN for encoding sequences | The encoder compresses the input sequence into hidden states |
| **Lesson 20**: Attention mechanism | Cross-attention lets the decoder look at *all* encoder states, not just the last one |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to explore attention in Seq2Seq!")

## 1. The Basic Encoder-Decoder (Without Attention)

First, let's understand the bottleneck problem.

In [ ]:
# Simple encoder: compress entire sentence into one vector
class SimpleEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        outputs, hidden = self.rnn(embedded)  # hidden: (1, batch, hidden_dim)
        return hidden  # Only return final hidden state!

# Simple decoder: generate from single context vector
class SimpleDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden):
        # x: (batch, 1) - one token at a time
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output)
        return prediction, hidden

print("The problem: entire input squeezed into one vector!")
print("")
print("Input sentence (10 words) --> [hidden_dim] vector --> Output sentence")
print("                              ^ BOTTLENECK! ^")

In [ ]:
# Visualize the bottleneck
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without attention
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Encoder words
encoder_words = ['The', 'cat', 'sat', 'on', 'the', 'mat']
for i, word in enumerate(encoder_words):
    ax.add_patch(plt.Rectangle((0.5, 8 - i*1.2), 2, 0.8, color='lightblue', ec='blue'))
    ax.text(1.5, 8.4 - i*1.2, word, ha='center', va='center', fontsize=10)

# Bottleneck
ax.add_patch(plt.Circle((5, 5), 0.8, color='red', alpha=0.5))
ax.text(5, 5, 'Context\nVector', ha='center', va='center', fontsize=9)

# Decoder words
decoder_words = ['Le', 'chat', 'assis', 'sur', 'le', 'tapis']
for i, word in enumerate(decoder_words):
    ax.add_patch(plt.Rectangle((7.5, 8 - i*1.2), 2, 0.8, color='lightgreen', ec='green'))
    ax.text(8.5, 8.4 - i*1.2, word, ha='center', va='center', fontsize=10)

# Arrows
ax.annotate('', xy=(4.2, 5), xytext=(2.5, 5), arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.annotate('', xy=(7.5, 5), xytext=(5.8, 5), arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_title('Without Attention: Information Bottleneck', fontsize=12)
ax.axis('off')

# With attention
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Encoder words
for i, word in enumerate(encoder_words):
    ax.add_patch(plt.Rectangle((0.5, 8 - i*1.2), 2, 0.8, color='lightblue', ec='blue'))
    ax.text(1.5, 8.4 - i*1.2, word, ha='center', va='center', fontsize=10)

# Decoder words
for i, word in enumerate(decoder_words):
    ax.add_patch(plt.Rectangle((7.5, 8 - i*1.2), 2, 0.8, color='lightgreen', ec='green'))
    ax.text(8.5, 8.4 - i*1.2, word, ha='center', va='center', fontsize=10)

# Attention connections
attention_pairs = [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5)]  # Simplified alignment
for src, tgt in attention_pairs:
    alpha = 0.8  # Attention weight
    ax.plot([2.5, 7.5], [8.4 - src*1.2, 8.4 - tgt*1.2], 
            color='purple', alpha=alpha, linewidth=2)

ax.set_title('With Attention: Direct Connections', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.show()

print("Attention lets the decoder 'look back' at relevant encoder states!")

## 2. Understanding Attention Weights

Attention answers: "Which input words should I focus on to generate this output word?"

The attention mechanism computes:
1. **Scores**: How relevant is each input word?
2. **Weights**: Softmax to get a probability distribution
3. **Context**: Weighted sum of encoder states

In [ ]:
def compute_attention(query, keys, values):
    """
    query: decoder hidden state (what we're looking for)
    keys: encoder hidden states (what we're searching through)
    values: encoder hidden states (what we retrieve)
    
    In basic attention, keys == values (encoder outputs)
    """
    # Step 1: Compute attention scores
    # Simple dot product: how similar is query to each key?
    scores = torch.matmul(query, keys.transpose(-2, -1))
    
    # Step 2: Normalize to get weights (probabilities)
    weights = F.softmax(scores, dim=-1)
    
    # Step 3: Weighted sum of values
    context = torch.matmul(weights, values)
    
    return context, weights

# Example
hidden_dim = 4
seq_len = 5

# Simulated encoder outputs (one for each input word)
encoder_outputs = torch.randn(1, seq_len, hidden_dim)

# Simulated decoder hidden state (what we're trying to decode)
decoder_hidden = torch.randn(1, 1, hidden_dim)

context, weights = compute_attention(decoder_hidden, encoder_outputs, encoder_outputs)

print(f"Encoder outputs shape: {encoder_outputs.shape}")
print(f"Decoder hidden shape: {decoder_hidden.shape}")
print(f"Attention weights: {weights.squeeze().detach().numpy().round(3)}")
print(f"Context vector shape: {context.shape}")
print(f"\nWeights sum to: {weights.sum().item():.3f} (always 1.0!)")

## 3. Visualizing Attention in Translation

Let's simulate a translation and visualize the attention pattern.

In [ ]:
# Simulated attention weights for English -> French translation
# Each row = which English words the French word attended to

source_words = ['The', 'black', 'cat', 'sat', 'on', 'the', 'mat', '<EOS>']
target_words = ['Le', 'chat', 'noir', 'etait', 'assis', 'sur', 'le', 'tapis', '<EOS>']

# Realistic attention pattern (words don't align perfectly!)
attention_matrix = np.array([
    [0.7, 0.1, 0.1, 0.0, 0.0, 0.1, 0.0, 0.0],  # Le -> The
    [0.1, 0.1, 0.7, 0.0, 0.0, 0.0, 0.1, 0.0],  # chat -> cat
    [0.0, 0.8, 0.1, 0.0, 0.0, 0.0, 0.1, 0.0],  # noir -> black (note: different order!)
    [0.0, 0.0, 0.1, 0.6, 0.2, 0.0, 0.1, 0.0],  # etait -> sat
    [0.0, 0.0, 0.0, 0.7, 0.2, 0.0, 0.1, 0.0],  # assis -> sat
    [0.0, 0.0, 0.0, 0.0, 0.8, 0.1, 0.1, 0.0],  # sur -> on
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.8, 0.1, 0.1],  # le -> the
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.8, 0.1],  # tapis -> mat
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.9],  # <EOS> -> <EOS>
])

plt.figure(figsize=(10, 8))
plt.imshow(attention_matrix, cmap='Blues', aspect='auto')
plt.colorbar(label='Attention Weight')

plt.xticks(range(len(source_words)), source_words, rotation=45, ha='right')
plt.yticks(range(len(target_words)), target_words)

plt.xlabel('Source (English)')
plt.ylabel('Target (French)')
plt.title('Attention Weights: English to French Translation')

# Add values in cells
for i in range(len(target_words)):
    for j in range(len(source_words)):
        if attention_matrix[i, j] > 0.3:
            plt.text(j, i, f'{attention_matrix[i, j]:.1f}', 
                    ha='center', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.show()

print("Key insight: 'noir' (black) attends to 'black' even though word order differs!")
print("English: 'black cat' vs French: 'chat noir' (cat black)")

## 4. Building an Attention Layer

There are several types of attention. The most common for Seq2Seq is **additive (Bahdanau) attention**.

In [ ]:
class BahdanauAttention(nn.Module):
    """
    Additive attention mechanism.
    score(s_t, h_i) = v^T * tanh(W_s * s_t + W_h * h_i)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_s = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
        
    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (batch, 1, hidden_dim)
        encoder_outputs: (batch, seq_len, hidden_dim)
        """
        # Expand decoder hidden to match encoder sequence length
        # (batch, 1, hidden) -> can broadcast with (batch, seq_len, hidden)
        
        # Compute attention scores
        scores = self.v(torch.tanh(
            self.W_s(decoder_hidden) + self.W_h(encoder_outputs)
        ))  # (batch, seq_len, 1)
        
        scores = scores.squeeze(-1)  # (batch, seq_len)
        
        # Softmax to get weights
        weights = F.softmax(scores, dim=-1)  # (batch, seq_len)
        
        # Compute context vector (weighted sum)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs)  # (batch, 1, hidden)
        
        return context, weights

# Test it
attention = BahdanauAttention(hidden_dim=64)

batch_size = 2
seq_len = 10
hidden_dim = 64

encoder_outputs = torch.randn(batch_size, seq_len, hidden_dim)
decoder_hidden = torch.randn(batch_size, 1, hidden_dim)

context, weights = attention(decoder_hidden, encoder_outputs)

print(f"Encoder outputs: {encoder_outputs.shape}")
print(f"Decoder hidden: {decoder_hidden.shape}")
print(f"Context vector: {context.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\nWeights for first example: {weights[0].detach().numpy().round(3)}")

## 5. Complete Seq2Seq with Attention

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)
        outputs, hidden = self.rnn(embedded)
        # Return ALL outputs (for attention) and final hidden state
        return outputs, hidden

class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(hidden_dim)
        # Input to RNN: embedding + context
        self.rnn = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden, encoder_outputs):
        """
        x: (batch, 1) - single token
        hidden: (1, batch, hidden) - decoder state
        encoder_outputs: (batch, src_len, hidden) - for attention
        """
        embedded = self.embedding(x)  # (batch, 1, embed)
        
        # Get attention context
        # Need to reshape hidden for attention
        hidden_for_attn = hidden.permute(1, 0, 2)  # (batch, 1, hidden)
        context, attn_weights = self.attention(hidden_for_attn, encoder_outputs)
        
        # Combine embedding and context
        rnn_input = torch.cat([embedded, context], dim=-1)
        
        # Run through RNN
        output, hidden = self.rnn(rnn_input, hidden)
        
        # Predict next token
        prediction = self.fc(output)
        
        return prediction, hidden, attn_weights

class Seq2SeqWithAttention(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, embed_dim, hidden_dim)
        self.decoder = AttentionDecoder(tgt_vocab_size, embed_dim, hidden_dim)
        
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        src: (batch, src_len) - source sequence
        tgt: (batch, tgt_len) - target sequence
        """
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.fc.out_features
        
        # Store outputs and attention weights
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size)
        attentions = []
        
        # Encode source
        encoder_outputs, hidden = self.encoder(src)
        
        # First input to decoder is <SOS> token (assume index 1)
        decoder_input = tgt[:, 0:1]  # (batch, 1)
        
        for t in range(1, tgt_len):
            output, hidden, attn_weights = self.decoder(decoder_input, hidden, encoder_outputs)
            outputs[:, t, :] = output.squeeze(1)
            attentions.append(attn_weights)
            
            # Teacher forcing: use actual target or prediction?
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher:
                decoder_input = tgt[:, t:t+1]
            else:
                decoder_input = output.argmax(dim=-1)
        
        return outputs, torch.stack(attentions, dim=1)

# Create model
model = Seq2SeqWithAttention(
    src_vocab_size=100,
    tgt_vocab_size=100,
    embed_dim=32,
    hidden_dim=64
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test the model
src = torch.randint(0, 100, (2, 8))   # Batch of 2, length 8
tgt = torch.randint(0, 100, (2, 10))  # Batch of 2, length 10

outputs, attentions = model(src, tgt)

print(f"Source shape: {src.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Output shape: {outputs.shape}")
print(f"Attentions shape: {attentions.shape}")
print(f"  (batch, tgt_len-1, src_len)")

## 6. A Simple Translation Example

Let's create a tiny toy translation task and visualize the attention.

In [ ]:
# Tiny vocabulary for number translation: English -> Spanish
en_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, 'one': 3, 'two': 4, 'three': 5, 'four': 6, 'five': 7}
es_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, 'uno': 3, 'dos': 4, 'tres': 5, 'cuatro': 6, 'cinco': 7}

en_idx_to_word = {v: k for k, v in en_vocab.items()}
es_idx_to_word = {v: k for k, v in es_vocab.items()}

# Training data: simple number translations
train_data = [
    ([3, 4, 5], [3, 4, 5]),      # one two three -> uno dos tres
    ([4, 5, 6], [4, 5, 6]),      # two three four -> dos tres cuatro
    ([3, 5, 7], [3, 5, 7]),      # one three five -> uno tres cinco
    ([6, 7, 3], [6, 7, 3]),      # four five one -> cuatro cinco uno
]

def prepare_batch(data):
    src_batch = []
    tgt_batch = []
    for src, tgt in data:
        src_batch.append(src + [2])  # Add <EOS>
        tgt_batch.append([1] + tgt + [2])  # Add <SOS> and <EOS>
    return torch.tensor(src_batch), torch.tensor(tgt_batch)

src_batch, tgt_batch = prepare_batch(train_data)
print(f"Source batch shape: {src_batch.shape}")
print(f"Target batch shape: {tgt_batch.shape}")
print(f"\nExample:")
print(f"  Source: {[en_idx_to_word[i.item()] for i in src_batch[0]]}")
print(f"  Target: {[es_idx_to_word[i.item()] for i in tgt_batch[0]]}")

In [ ]:
# Train on this tiny dataset
model = Seq2SeqWithAttention(
    src_vocab_size=8,
    tgt_vocab_size=8,
    embed_dim=16,
    hidden_dim=32
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

losses = []
for epoch in range(200):
    optimizer.zero_grad()
    
    outputs, attentions = model(src_batch, tgt_batch, teacher_forcing_ratio=0.5)
    
    # Reshape for loss
    output_flat = outputs[:, 1:, :].reshape(-1, 8)
    target_flat = tgt_batch[:, 1:].reshape(-1)
    
    loss = criterion(output_flat, target_flat)
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# Visualize attention for a translation
model.train(False)
with torch.no_grad():
    test_src = src_batch[0:1]  # "one two three"
    test_tgt = tgt_batch[0:1]  # "<SOS> uno dos tres <EOS>"
    
    outputs, attentions = model(test_src, test_tgt, teacher_forcing_ratio=0)
    
    # Get predictions
    predictions = outputs.argmax(dim=-1)[0]
    
    print("Translation:")
    print(f"  Source: {[en_idx_to_word[i.item()] for i in test_src[0]]}")
    print(f"  Predicted: {[es_idx_to_word[i.item()] for i in predictions[1:]]}")
    print(f"  Target: {[es_idx_to_word[i.item()] for i in test_tgt[0][1:]]}")

# Plot attention
attn_matrix = attentions[0].squeeze().numpy()  # (tgt_len-1, src_len)

src_words = [en_idx_to_word[i.item()] for i in test_src[0]]
tgt_words = [es_idx_to_word[i.item()] for i in predictions[1:]]

plt.figure(figsize=(8, 6))
plt.imshow(attn_matrix, cmap='Blues', aspect='auto')
plt.colorbar(label='Attention Weight')

plt.xticks(range(len(src_words)), src_words, rotation=45, ha='right')
plt.yticks(range(len(tgt_words)), tgt_words)

plt.xlabel('Source (English)')
plt.ylabel('Target (Spanish)')
plt.title('Learned Attention Pattern')

plt.tight_layout()
plt.show()

print("\nThe model learned to attend to corresponding words!")

## 7. Why Attention Matters

Attention was revolutionary because it:

1. **Solves the bottleneck**: No need to compress everything into one vector
2. **Handles long sequences**: Can attend to any position, not just recent ones
3. **Provides interpretability**: Attention weights show what the model focuses on
4. **Enables parallelization**: Led directly to the Transformer architecture!

In [ ]:
# Compare: information flow with and without attention
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without attention: sequential bottleneck
ax = axes[0]
positions = np.arange(10)
info_retention = np.exp(-0.3 * positions[::-1])  # Exponential decay

ax.bar(positions, info_retention, color='coral', alpha=0.7)
ax.set_xlabel('Input Position')
ax.set_ylabel('Information Retained')
ax.set_title('Without Attention: Information Fades')
ax.set_xticks(positions)
ax.set_xticklabels([f'w{i}' for i in range(10)])

# With attention: direct access
ax = axes[1]
info_retention = np.ones(10)  # Can access any position
attention_weights = np.array([0.1, 0.1, 0.5, 0.1, 0.05, 0.05, 0.02, 0.02, 0.03, 0.02])

ax.bar(positions, info_retention, color='lightblue', alpha=0.5, label='Available')
ax.bar(positions, attention_weights * 2, color='blue', alpha=0.7, label='Attended')
ax.set_xlabel('Input Position')
ax.set_ylabel('Information Access')
ax.set_title('With Attention: Direct Access to Any Position')
ax.set_xticks(positions)
ax.set_xticklabels([f'w{i}' for i in range(10)])
ax.legend()

plt.tight_layout()
plt.show()

print("Without attention: Later words in decoder can barely 'see' early input words")
print("With attention: Decoder can directly access any input word at any time!")

## Check Your Understanding

1. What is the "bottleneck problem" in basic Seq2Seq models?
2. What do attention weights represent?
3. Why is attention helpful for translating between languages with different word orders?
4. What is teacher forcing, and why do we use it during training?
5. How does attention improve interpretability?

In [ ]:
# --- Quick Check: Encoder vs Decoder ---
# In a seq2seq model, what is the encoder's job?
# a) Generate the output sequence word by word
# b) Compress the input sequence into hidden state representations
# c) Compute attention weights
# d) Select the best translation from candidates

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "The encoder reads the input and creates hidden representations. The decoder generates output using those representations."
print("Exercise 1 passed! ✓")

# --- Quick Check: Without Attention ---
# Without attention, what information does the decoder have from the encoder?
# a) All hidden states from every encoder step
# b) Only the final hidden state (the bottleneck)
# c) Only the input embeddings
# d) Nothing — it generates independently

your_answer_2 = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer_2 is not None, "Pick an answer!"
assert your_answer_2 == 'b', "Without attention, the entire input is squeezed into ONE vector (the last hidden state). That's the bottleneck!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## Summary

**Sequence-to-Sequence with Attention** revolutionized machine translation:

- **Encoder-Decoder architecture**: Encoder reads input, decoder generates output
- **Attention mechanism**: Decoder can "look back" at relevant encoder states
- **Attention weights**: Soft alignment between input and output positions
- **Context vector**: Weighted combination of encoder outputs

Key formulas:
```
scores = attention_function(decoder_state, encoder_states)
weights = softmax(scores)  # Sum to 1
context = weighted_sum(weights, encoder_states)
```

This attention mechanism was so successful that researchers asked: "What if we used attention everywhere?" That question led to the **Transformer** architecture.

**Next up**: Self-Attention - when words attend to each other in the same sequence!